# 09. 피처셋 계약 랩

## 연구 질문

> 노트북 06~08에서 검증한 피처 조합을 **설정값(config)으로 표현**했을 때,
> 승격 전과 같은 결과를 내면서도 재현성 계약이 유지되는가?

노트북 06~08은 피처를 하나씩 추가하며 2024 holdout 점수를 올렸다. 그러나 그 개선분은
노트북 안에만 있었고 `src/baram/train.py`가 만드는 실제 제출물에는 반영되지 않았다.
이 노트북은 승격의 첫 단계인 **피처셋 config 계층**이 옳게 설계됐는지를 닫는다.

점수를 재는 노트북이 아니다. 점수 재현은 11번 노트북이 담당한다.
여기서 확인하는 것은 "설정을 바꿨을 때 시스템이 정직하게 반응하는가"다.

## 목차

1. 승격 전후를 가르는 것은 무엇인가
2. 프리셋 4종 — 랩 실험을 config로 옮기기 (Decision Box ①②)
3. `official_mean`은 legacy production과 컬럼·값이 완전히 같다
4. YAML 오버라이드 — 유연성과 검증 표면의 균형 (Decision Box ③④)
5. 전처리 계약은 이제 config에서 파생된다 (Decision Box ⑤)
6. 종합 결론

## 이 노트북의 전제

- 공식 원자료는 Git에 커밋하지 않으므로 로컬 `data/raw/open/`에서 읽는다.
- 모델 파일, 제출 CSV, registry 행은 만들지 않는다. 계약 검증만 수행한다.
- 설계 근거는 `docs/design/05-feature-set-promotion.md` v1.0을 따른다.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd


def resolveProjectRoot():
  """worktree/일반 체크아웃 어디서 실행하든 src와 원자료를 찾는다."""
  here = Path.cwd().resolve()
  for candidate in [here, *here.parents]:
    if (candidate / "src" / "baram").is_dir():
      return candidate
  raise RuntimeError("src/baram을 찾지 못했습니다")


def resolveOfficialDataDir(projectRoot):
  for candidate in [projectRoot, *projectRoot.parents]:
    dataDir = candidate / "data" / "raw" / "open"
    if (dataDir / "train" / "train_labels.csv").is_file():
      return dataDir
  raise RuntimeError("공식 데이터 디렉터리를 찾지 못했습니다")


projectRoot = resolveProjectRoot()
if str(projectRoot / "src") not in sys.path:
  sys.path.insert(0, str(projectRoot / "src"))
dataDir = resolveOfficialDataDir(projectRoot)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"project root : {projectRoot.name}")
print(f"official data: {dataDir}")

project root : codex-to-claude-transition-a9327d
official data: C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\data\raw\open


프로젝트 루트와 공식 데이터 위치를 먼저 고정한다. worktree에는 원자료가 없으므로
상위 경로를 거슬러 올라가며 찾는다. 이 탐색이 실패하면 이후 셀은 실행되지 않으므로,
데이터 없이 만들어진 결론이 노트북에 남는 일은 없다.

---

## 1. 승격 전후를 가르는 것은 무엇인가

승격 전 상태를 숫자로 먼저 확인한다. 노트북 랩이 올린 점수와, production이 실제로
쓰는 피처 사이의 간격이 이 작업의 출발점이다.

In [2]:
labSummary = pd.DataFrame(
  [
    ("02 공식 baseline", "grid mean", 74, 0.576938, "통제군"),
    ("06 weather grid", "+ std/min/max + lead", 271, 0.585265, "+0.008327"),
    ("07 wind vector", "+ speed/wind-from", 319, 0.586482, "+0.009544"),
    ("08 turbine spatial", "+ IDW p=2 all-group", 550, 0.592485, "+0.015547"),
    ("08 turbine spatial", "+ nearest own-group", 396, 0.589828, "+0.012890"),
  ],
  columns=["노트북", "추가한 피처", "features", "2024 holdout total", "통제군 대비"],
)
labSummary

,노트북,추가한 피처,features,2024 holdout total,통제군 대비
0,02 공식 baseline,grid mean,74,0.576938,통제군
1,06 weather grid,+ std/min/max + lead,271,0.585265,+0.008327
2,07 wind vector,+ speed/wind-from,319,0.586482,+0.009544
3,08 turbine spatial,+ IDW p=2 all-group,550,0.592485,+0.015547
4,08 turbine spatial,+ nearest own-group,396,0.589828,+0.012890


### 1-1. 랩은 총 +0.0155를 올렸지만 production은 한 걸음도 움직이지 않았다

위 표의 개선분은 전부 노트북 셀 안에서 계산됐다. `src/baram/baseline.py`의
`aggregate_weather()`는 여전히 `groupby("forecast_kst_dtm").mean()` 한 줄이고,
`train.py`를 실행하면 74개(+lead 2개) 피처짜리 공식 baseline이 나온다.

승격을 막고 있던 것은 게으름이 아니라 **구조**였다.

| 장애물 | 내용 |
|--------|------|
| 이름 충돌 | `build_weather_features`가 `baram.baseline`과 `baram.features.weather_grid`에 각각 있고 계약이 다르다 |
| 로더 부재 | `info.xlsx` → 터빈 좌표 변환이 노트북 08 셀 안에만 인라인으로 존재했다 |
| 계약 하드코딩 | `PREPROCESSING_CONTRACT`에 `statistics: ["mean"]`이 상수로 박혀 있었다 |
| bundle 단일 목록 | own-group은 target마다 컬럼이 다른데 bundle은 목록 하나를 공유했다 |

셋째 항목이 특히 위험하다. 피처를 바꾸고 상수를 그대로 두면 **전처리가 달라졌는데
`preprocessing_sha256`는 같아진다.** 재현성 레이어 전체가 거짓 신호를 내게 된다.

---

## 2. 프리셋 4종 — 랩 실험을 config로 옮기기

랩에서 비교한 후보는 6개였지만 프리셋으로 승격한 것은 4개다.

In [3]:
from baram.feature_config import (
  DEFAULT_FEATURE_SET,
  FEATURE_SETS,
  feature_set_canonical_json,
  feature_set_sha256,
  get_feature_set,
)

presetRows = []
for name, config in FEATURE_SETS.items():
  spatial = config.spatial
  presetRows.append(
    {
      "preset": name,
      "statistics": "/".join(config.statistics),
      "lead": config.include_lead,
      "vector": config.wind_vector,
      "pooling": "-" if spatial is None else "/".join(spatial.methods),
      "idw_power": None if spatial is None else spatial.idw_power,
      "scope": "-" if spatial is None else spatial.scope,
      "config_sha256": feature_set_sha256(config)[:12],
    }
  )
presetTable = pd.DataFrame(presetRows).set_index("preset")
print(f"기본 피처셋: {DEFAULT_FEATURE_SET}")
presetTable

기본 피처셋: official_mean


,statistics,lead,vector,pooling,idw_power,scope,config_sha256
preset,,,,,,,
official_mean,mean,True,False,-,NaN,-,c986efb8ab55
expanded_vector,mean/std/min/max,True,True,-,NaN,-,3adc1340a4a6
spatial_idw2_all_group,mean/std/min/max,True,True,idw,2.0,all_group,241bcf22fa05
spatial_nearest_own_group,mean/std/min/max,True,True,nearest,2.0,own_group,ab77253531ca


### Decision Box ① — 기본값은 랩 통제군이 아니라 production 경로를 따른다

**선택지**

- (A) `official_mean` = 노트북 06~08의 통제군 (mean만, lead 없음, 74개)
- (B) `official_mean` = `baram.baseline`의 legacy 경로 (mean + lead, 76개)

**근거**

기본값이 존재하는 이유는 "인자 없이 실행하면 승격 전과 같은 결과가 나온다"를 보장하기
위해서다. 그런데 legacy `aggregate_weather()`는 lead feature를 **항상** 붙인다.
(A)를 고르면 `python -m baram.train --run`의 결과가 조용히 바뀐다.

랩 통제군이 74개였던 것은 랩의 선택이었지, production의 정의가 아니다.

**채택: (B)** — `include_lead=True`. 대신 랩 통제군(74개)과는 lead 2개만큼 다르다는
사실을 명시하고, 11번 노트북의 점수 체인에서는 ad-hoc config로 재현한다.

### Decision Box ② — 프리셋은 4종으로 한정한다

**선택지**

- (A) 3종 — 통제군 + 채택 후보만
- (B) 4종 — + own-group nearest 축소 대안
- (C) 6종 — 랩에서 비교한 후보 전부

**근거**

`vector_plus_idw_p1_all`과 `vector_plus_idw_p2_own`은 랩에서 채택되지 않았고,
프리셋으로 올리면 재현 검증과 테스트 표면만 늘어난다. 반면 own-group nearest는
"차원을 154개 줄이고도 +0.0033을 유지한 강건성 대안"으로 남겨둔 후보라, 다음 작업인
seasonal fold에서 바로 필요하다.

**채택: (B)** — 4종. 탈락한 2종은 4절의 YAML 오버라이드로 언제든 재현할 수 있다.

---

## 3. `official_mean`은 legacy production과 컬럼·값이 완전히 같다

Decision Box ①이 말로만 옳은지, 공식 데이터로 확인한다. 이것이 G2 골든 게이트의 핵심이다.

In [4]:
def readOfficial(relativePath):
  return pd.read_csv(dataDir / relativePath, encoding="utf-8-sig")


trainLabels = readOfficial("train/train_labels.csv")
trainLabels["kst_dtm"] = pd.to_datetime(trainLabels["kst_dtm"])
ldapsTrain = readOfficial("train/ldaps_train.csv")
gfsTrain = readOfficial("train/gfs_train.csv")

print(f"train_labels : {trainLabels.shape}")
print(f"ldaps_train  : {ldapsTrain.shape}")
print(f"gfs_train    : {gfsTrain.shape}")

train_labels : (26304, 4)
ldaps_train  : (420864, 35)
gfs_train    : (236736, 40)


공식 학습 입력을 읽었다. 행 수는 00번 감사 노트북에서 잠근 값과 같다
(label 26,304 / LDAPS 420,864 / GFS 236,736).

이제 legacy 경로와 새 파이프라인을 같은 입력에 대해 각각 돌린다.

In [5]:
from baram.baseline import build_training_features
from baram.feature_pipeline import build_train_features

legacyFeatures = build_training_features(trainLabels, ldapsTrain, gfsTrain)
promotedBuild = build_train_features(
  get_feature_set("official_mean"),
  time_index=trainLabels["kst_dtm"],
  ldaps=ldapsTrain,
  gfs=gfsTrain,
)

sameColumns = list(legacyFeatures.X.columns) == list(promotedBuild.matrix.columns)
valuesEqual = legacyFeatures.X.reset_index(drop=True).equals(
  promotedBuild.matrix.reset_index(drop=True)
)

pd.DataFrame(
  [
    ("legacy build_training_features", *legacyFeatures.X.shape),
    ("promoted official_mean", *promotedBuild.matrix.shape),
  ],
  columns=["경로", "rows", "features"],
).assign(컬럼_동일=sameColumns, 값_동일=valuesEqual)

,경로,rows,features,컬럼_동일,값_동일
0,legacy build_training_features,26304,76,True,True
1,promoted official_mean,26304,76,True,True


### 3-1. 두 경로는 컬럼 이름·순서·값이 모두 같다

`26,304 × 76`으로 형태가 같고, 컬럼 목록과 값이 전부 일치한다.
즉 **`--feature-set`을 생략하면 승격 전 제출물과 바이트 단위로 같은 결과**가 나온다.

이 등가성이 롤백 3중 장치의 첫 번째 층이다. 뒤 작업에서 무언가 잘못돼도
CLI 인자만 빼면 기준선으로 복귀한다.

컬럼 배치도 눈으로 확인해 둔다.

In [6]:
promotedColumns = list(promotedBuild.matrix.columns)
layout = pd.DataFrame(
  [
    ("calendar", 9, promotedColumns[0], promotedColumns[8]),
    (
      "ldaps 집계",
      sum(name.startswith("ldaps_") for name in promotedColumns),
      promotedColumns[9],
      "ldaps_lead_hour",
    ),
    (
      "gfs 집계",
      sum(name.startswith("gfs_") for name in promotedColumns),
      next(name for name in promotedColumns if name.startswith("gfs_")),
      promotedColumns[-1],
    ),
  ],
  columns=["구획", "컬럼 수", "첫 컬럼", "마지막 컬럼"],
)
layout

,구획,컬럼 수,첫 컬럼,마지막 컬럼
0,calendar,9,month,month_cos
1,ldaps 집계,31,ldaps_heightAboveGround_10_10u_mean,ldaps_lead_hour
2,gfs 집계,36,gfs_heightAboveGround_10_10u_mean,gfs_lead_hour


`calendar → ldaps → gfs` 순서이고 각 source의 마지막이 `lead_hour`다.
이 순서는 설계서 4.5절의 조립 계약이며, 바뀌면 `max_features="sqrt"`인 RandomForest가
다른 트리를 만들어 점수가 재현되지 않는다. 순서 자체를 테스트로 잠가 둔 이유다.

---

## 4. YAML 오버라이드 — 유연성과 검증 표면의 균형

프리셋만으로는 "IDW p=1로 한 번만 돌려보고 싶다" 같은 요구를 못 받는다.
그렇다고 YAML로 config 전체를 정의하게 하면 검증할 입력 공간이 폭발한다.

In [7]:
import tempfile

from baram.feature_config import resolve_feature_set

scratch = Path(tempfile.mkdtemp(prefix="baram_feature_set_"))

overrideYaml = """
# 랩에서 탈락한 IDW p=1 후보를 프리셋 수정 없이 재현한다.
extends: spatial_idw2_all_group
overrides:
  spatial:
    idw_power: 1.0
"""
overridePath = scratch / "idw_p1.yaml"
overridePath.write_text(overrideYaml, encoding="utf-8")

resolvedOverride = resolve_feature_set(config_path=overridePath)
baseConfig = get_feature_set("spatial_idw2_all_group")

pd.DataFrame(
  [
    ("상속 프리셋", baseConfig.name, "/".join(baseConfig.spatial.methods), baseConfig.spatial.idw_power, baseConfig.spatial.scope),
    (
      "YAML 결과",
      resolvedOverride.config.name,
      "/".join(resolvedOverride.config.spatial.methods),
      resolvedOverride.config.spatial.idw_power,
      resolvedOverride.config.spatial.scope,
    ),
  ],
  columns=["구분", "name", "methods", "idw_power", "scope"],
)

,구분,name,methods,idw_power,scope
0,상속 프리셋,spatial_idw2_all_group,idw,2.0,all_group
1,YAML 결과,spatial_idw2_all_group+idw_p1,idw,1.0,all_group


`idw_power`만 덮어썼고 `methods`와 `scope`는 상속 프리셋 값이 그대로 남았다.
`spatial`만 부분 병합하고 나머지 필드는 전체 치환하는 규약대로 동작한다.

### Decision Box ③ — YAML은 `extends` 상속과 부분 오버라이드로만 허용한다

**선택지**

- (A) 코드 내 named preset만
- (B) YAML로 config를 처음부터 정의
- (C) preset을 `extends`로 상속하고 일부만 덮어쓰기

**근거**

(A)는 가장 안전하지만 프리셋에 없는 조합을 시도하려면 코드를 고쳐야 한다.
(B)는 유연하지만 필드 조합 전체가 검증 대상이 되고, 오타 한 글자가 조용히 다른
전처리를 만든다. (C)는 항상 검증된 프리셋에서 출발하므로 잘못될 수 있는 범위가
오버라이드한 필드로 제한된다.

**채택: (C)** — `extends`는 필수, 알 수 없는 키는 무시하지 않고 즉시 실패.

아래에서 실패 계약을 직접 확인한다.

In [8]:
badConfigs = {
  "extends 누락": ("missing_extends", "overrides:\n  include_lead: true\n"),
  "최상위 오타 키": ("unknown_top_key", "extends: expanded_vector\nextra: 1\n"),
  "override 오타 키": ("unknown_override_key", "extends: expanded_vector\noverrides:\n  includelead: true\n"),
  "spatial 오타 키": ("unknown_spatial_key", "extends: spatial_idw2_all_group\noverrides:\n  spatial:\n    power: 1.0\n"),
  "없는 프리셋 상속": ("unknown_preset", "extends: nope\n"),
  "매핑이 아님": ("not_a_mapping", "- extends: expanded_vector\n"),
}

guardRows = []
for label, (stem, text) in badConfigs.items():
  path = scratch / f"{stem}.yaml"
  path.write_text(text, encoding="utf-8")
  try:
    resolve_feature_set(config_path=path)
    guardRows.append((label, "통과", "계약 위반이 감지되지 않음"))
  except (ValueError, TypeError) as error:
    guardRows.append((label, "거부", str(error)[:70]))

pd.DataFrame(guardRows, columns=["잘못된 입력", "결과", "메시지"]).set_index("잘못된 입력")

,결과,메시지
잘못된 입력,,
extends 누락,거부,피처셋 config에는 상속할 프리셋 이름을 extends로 지정해야 합니다
최상위 오타 키,거부,피처셋 config에 알 수 없는 키: ['extra']; 허용값=['extends...
override 오타 키,거부,overrides에 알 수 없는 키: ['includelead']; 허용값=['ca...
spatial 오타 키,거부,spatial 오버라이드에 알 수 없는 키: ['power']; 허용값=['idw_...
없는 프리셋 상속,거부,"알 수 없는 피처셋 이름: 'nope'; 허용값=['expanded_vector',..."
매핑이 아님,거부,피처셋 config는 최상위가 매핑이어야 합니다: C:\Users\kik32\App...


### 4-1. 오타는 조용히 무시되지 않고 전부 거부된다

여섯 가지 잘못된 입력이 모두 명시적 예외로 끝났다. `includelead: true`처럼
한 글자 틀린 키를 무시했다면, 사용자는 lead를 껐다고 믿지만 실제로는 켜진 상태로
모델이 학습되고 그 사실이 어디에도 남지 않는다. 재현성 관점에서 가장 나쁜 실패다.

### Decision Box ④ — config hash는 이름을 제외하고 계산한다

**선택지**

- (A) `name`을 포함한 전체 config를 hash
- (B) `name`을 제외한 전처리 의미만 hash

**근거**

YAML로 해석한 config는 이름이 `expanded_vector+idw_p1`처럼 파일명을 물고 온다.
(A)를 고르면 **같은 전처리인데 파일명만 달라도 다른 hash**가 나온다. hash가 식별해야
하는 것은 라벨이 아니라 "어떤 전처리를 했는가"다.

**채택: (B)** — 아래에서 정규화가 실제로 포맷 차이를 흡수하는지 sweep한다.

In [9]:
formatVariants = {
  "기본": ("plain", "extends: expanded_vector\n"),
  "주석 포함": ("commented", "# 설명\nextends: expanded_vector\n"),
  "여분 공백": ("spaced", "extends:    expanded_vector\n\n\n"),
  "따옴표": ("quoted", 'extends: "expanded_vector"\n'),
}

sweepRows = []
for label, (stem, text) in formatVariants.items():
  path = scratch / f"{stem}.yaml"
  path.write_text(text, encoding="utf-8")
  resolved = resolve_feature_set(config_path=path)
  sweepRows.append((label, resolved.config.name, resolved.sha256[:16]))

presetDigest = feature_set_sha256(get_feature_set("expanded_vector"))
sweepRows.append(("프리셋 직접", "expanded_vector", presetDigest[:16]))

sweepTable = pd.DataFrame(sweepRows, columns=["YAML 표기", "해석된 name", "config_sha256"])
sweepTable["프리셋과 동일"] = sweepTable["config_sha256"] == presetDigest[:16]
sweepTable

,YAML 표기,해석된 name,config_sha256,프리셋과 동일
0,기본,expanded_vector+plain,3adc1340a4a6a5d0,True
1,주석 포함,expanded_vector+commented,3adc1340a4a6a5d0,True
2,여분 공백,expanded_vector+spaced,3adc1340a4a6a5d0,True
3,따옴표,expanded_vector+quoted,3adc1340a4a6a5d0,True
4,프리셋 직접,expanded_vector,3adc1340a4a6a5d0,True


### 4-2. 이름은 달라져도 hash는 하나로 모인다

네 가지 YAML 표기와 프리셋 직접 참조가 **모두 같은 hash**를 냈다. 해석된 `name`은
파일명에 따라 달랐지만 전처리 의미가 같으므로 hash도 같다.

정규화 규칙은 다음과 같다.

| 규칙 | 값 |
|------|-----|
| 직렬화 | `json.dumps(sort_keys=True, ensure_ascii=False, separators=(",", ":"))` |
| float | `format(value, ".10g")` 문자열로 고정 |
| tuple | list로 변환하되 순서 보존 |
| 제외 필드 | `name` |

실제 직렬화 결과를 눈으로 확인한다.

In [10]:
print(feature_set_canonical_json(get_feature_set("spatial_nearest_own_group")))

{"calendar":true,"include_lead":true,"spatial":{"idw_power":"2","methods":["nearest"],"scope":"own_group"},"statistics":["mean","std","min","max"],"wind_vector":true}


`name`이 없고 키가 알파벳 순으로 정렬돼 있으며, `idw_power`가 `"2"` 문자열이다.
플랫폼별 float repr 차이(`2.0` vs `2`)가 hash를 흔들지 않게 하려는 조치다.

---

## 5. 전처리 계약은 이제 config에서 파생된다

가장 위험했던 하드코딩을 걷어낸다. 승격 작업에서 이 부분을 빼먹으면 나머지가
모두 옳아도 재현성 기록이 거짓말을 하게 된다.

### Decision Box ⑤ — 계약을 파생시키되 legacy 상수는 골든 기준으로 남긴다

**선택지**

- (A) `PREPROCESSING_CONTRACT` 상수를 config 파생 결과로 교체하고 상수는 삭제
- (B) 상수를 남기고, `official_mean` 파생 결과가 상수와 같은지 테스트로 잠금

**근거**

(A)는 깔끔하지만 "파생이 legacy와 같다"를 검증할 기준점이 사라진다. 파생 로직에
버그가 들어가도 비교 대상이 없어 알아챌 수 없다. (B)는 상수가 중복처럼 보이지만
실제로는 **동결된 기대값** 역할을 한다.

**채택: (B)**

In [11]:
from baram.registry import (
  MODEL_METADATA_SCHEMA_VERSION,
  PREPROCESSING_CONTRACT,
  build_preprocessing_contract,
)
from baram.reproducibility import sha256_text, stable_json_dumps

derivedOfficial = build_preprocessing_contract(get_feature_set("official_mean"))
goldenEqual = derivedOfficial == PREPROCESSING_CONTRACT
goldenHashEqual = sha256_text(stable_json_dumps(derivedOfficial)) == sha256_text(
  stable_json_dumps(PREPROCESSING_CONTRACT)
)

print(f"metadata schema version : {MODEL_METADATA_SCHEMA_VERSION}")
print(f"official_mean 파생 == legacy 상수 : {goldenEqual}")
print(f"계약 hash 동일 : {goldenHashEqual}")

metadata schema version : 1.1
official_mean 파생 == legacy 상수 : True
계약 hash 동일 : True


### 5-1. `official_mean`의 파생 계약은 legacy 상수와 바이트 단위로 같다

G2 골든이 통과했다. 파생으로 바꿨지만 기본 경로의 `preprocessing_sha256`는
승격 전 값을 그대로 유지한다.

이제 프리셋을 바꿨을 때 계약이 **실제로 달라지는지**가 남았다.

In [12]:
contractRows = []
for name, config in FEATURE_SETS.items():
  contract = build_preprocessing_contract(config)
  contractRows.append(
    {
      "preset": name,
      "statistics": len(contract["weather_aggregation"]["statistics"]),
      "lead": "lead_feature" in contract,
      "wind_vector": "wind_vector" in contract,
      "spatial": "spatial_pooling" in contract,
      "scope": contract.get("spatial_pooling", {}).get("scope", "-"),
      "preprocessing_sha256": sha256_text(stable_json_dumps(contract))[:12],
    }
  )
contractTable = pd.DataFrame(contractRows).set_index("preset")
contractTable["legacy와 동일"] = contractTable["preprocessing_sha256"] == sha256_text(
  stable_json_dumps(PREPROCESSING_CONTRACT)
)[:12]
contractTable

,statistics,lead,wind_vector,spatial,scope,preprocessing_sha256,legacy와 동일
preset,,,,,,,
official_mean,1,True,False,False,-,0bf7481d7482,True
expanded_vector,4,True,True,False,-,e616efebf501,False
spatial_idw2_all_group,4,True,True,True,all_group,3a2a19792e66,False
spatial_nearest_own_group,4,True,True,True,own_group,075aca5688e8,False


### 5-2. 전처리가 달라지면 계약 hash도 반드시 달라진다

네 프리셋의 `preprocessing_sha256`가 모두 다르고, `official_mean`만 legacy와 같다.
`spatial_idw2_all_group`과 `spatial_nearest_own_group`은 통계·lead·vector 설정이
동일하지만 pooling method와 scope가 달라 hash가 갈렸다.

승격 전이었다면 이 네 경우가 **전부 같은 hash**를 기록했을 것이다.

마지막으로 계약에 실제로 무엇이 적히는지 확인한다.

In [13]:
spatialContract = build_preprocessing_contract(get_feature_set("spatial_idw2_all_group"))
print(json.dumps(
  {key: spatialContract[key] for key in ("wind_vector", "spatial_pooling")},
  ensure_ascii=False,
  indent=2,
))

{
  "wind_vector": {
    "derived_from": "raw grid rows before aggregation",
    "components": [
      "speed",
      "wind_from_sin",
      "wind_from_cos"
    ],
    "specs": {
      "gfs": [
        "10m",
        "80m",
        "100m"
      ],
      "ldaps": [
        "10m"
      ]
    }
  },
  "spatial_pooling": {
    "methods": [
      "idw"
    ],
    "idw_power": 2.0,
    "scope": "all_group",
    "weights": "haversine distance, turbine capacity weighted within group",
    "fit_scope": "training weather grid geometry"
  }
}


wind vector 파생 시점("집계 이전 raw grid 행")과 공간 pooling의 method·지수·scope·
가중 방식·fit 범위가 계약에 남는다. 나중에 이 모델이 어떤 전처리로 만들어졌는지
묻는 질문은 sidecar 하나로 답할 수 있다.

---

## 6. 종합 결론

### 6-1. 연구 질문

> 랩에서 검증한 피처 조합을 config로 표현했을 때, 승격 전과 같은 결과를 내면서도
> 재현성 계약이 유지되는가?

**유지된다.** 단, 세 가지 설계 결정이 함께 있어야 성립한다 —
기본값을 production 경로에 맞추고, hash에서 이름을 빼고, 계약을 파생시키는 것.

### 6-2. 단계별 요약

| 절 | 확인한 것 | 결과 |
|----|-----------|------|
| 1 | 승격 전 격차 | 랩 +0.0155가 production에 0% 반영 |
| 2 | 프리셋 4종 | 통제군 2 + 채택 후보 1 + 축소 대안 1 |
| 3 | legacy 등가성 | 26,304 × 76 컬럼·값 완전 동일 |
| 4 | YAML 오버라이드 | 오타 6종 전부 거부, 표기 4종이 같은 hash |
| 5 | 계약 파생 | official_mean 골든 통과, 프리셋별 hash 분리 |

### 6-3. 주요 발견

1. **랩 통제군과 production baseline은 원래 달랐다.** 랩의 74개에는 lead feature가
   없고 production의 76개에는 있다. 기본값을 랩 쪽에 맞췄다면 인자 없는 실행 결과가
   조용히 바뀌었을 것이다.
2. **hash에 이름을 넣으면 안 된다.** 같은 전처리를 YAML로 부르든 프리셋으로 부르든
   같은 hash가 나와야 실험 대조가 성립한다.
3. **하드코딩 계약이 가장 위험했다.** 네 프리셋이 전부 같은 `preprocessing_sha256`를
   기록하는 상태였고, 이는 재현성 레이어가 있는데도 없는 것보다 나쁜 상황이다.

### 6-4. 시사점

기본값 보존과 골든 테스트를 함께 두면 "새 기능을 켜지 않으면 아무것도 바뀌지 않는다"를
코드가 아니라 **검증으로** 보증할 수 있다. 승격처럼 되돌릴 수 있어야 하는 작업에서는
이 조합이 브랜치 전략보다 실효적이다.

### 6-5. 한계

- 이 노트북은 **점수를 재지 않는다.** 조립 결과가 랩 점수를 재현하는지는 11번에서 닫는다.
- `official_mean` 외 프리셋의 legacy 등가성은 정의상 존재하지 않는다. 비교 대상은
  랩 노트북의 feature 개수와 점수뿐이다.
- YAML 오버라이드의 검증은 키 이름과 타입까지다. 의미적으로 무의미한 조합
  (예: `methods=("nearest",)`인데 `idw_power` 지정)은 거부하지 않고 통과시킨다.

### 6-6. 요약

피처셋 config 계층은 **승격 전 동작을 바꾸지 않으면서** 랩의 네 후보를 CLI 한 줄로
선택할 수 있게 만들었고, 전처리가 달라지면 계약 hash가 반드시 따라 달라진다.
다음 노트북(10)은 이 config가 만드는 feature matrix의 조립 순서와 train/test 대칭을 닫는다.

---

## 산출물 안전 확인

이 노트북은 모델·제출물·registry를 만들지 않는다. 임시 YAML만 시스템 임시 폴더에 남긴다.

In [14]:
createdFiles = sorted(path.name for path in scratch.glob("*"))
print(f"임시 YAML 위치 : {scratch}")
print(f"임시 파일 수   : {len(createdFiles)}")
print("프로젝트 내부 산출물 생성 여부 :", any(
  (projectRoot / name).exists() for name in ("outputs", "submission.csv", "baseline.pkl")
))

임시 YAML 위치 : C:\Users\kik32\AppData\Local\Temp\baram_feature_set_2rw1fuzv
임시 파일 수   : 11
프로젝트 내부 산출물 생성 여부 : False


프로젝트 안에는 아무 산출물도 만들지 않았다. 계약 검증만 수행하고 종료한다.